In [5]:
from pathlib import Path
import ast
import csv
import json


def mejores_configuraciones(csv_path, metricas=None, grupo="measure"):
    """Muestra las configuraciones con el mejor valor de cada métrica.

    Parámetros:
        csv_path: ruta a cualquier CSV dentro de config_results.
        metricas: columnas numéricas a maximizar. Por defecto, todas las
                  columnas numéricas salvo las que forman parte de la configuración.
        grupo: columna que separa las métricas/distancias, normalmente ``measure``.
    """
    csv_path = Path(csv_path)

    with csv_path.open("r", newline="", encoding="utf-8") as archivo:
        filas = list(csv.DictReader(archivo))

    if not filas:
        raise ValueError(f"El archivo no contiene resultados: {csv_path}")

    columnas = list(filas[0])
    if grupo not in columnas:
        raise ValueError(f"No existe la columna de agrupación {grupo!r} en el CSV")

    # Detectar automáticamente las columnas de resultados (map1, map5, accuracy, ...).
    if metricas is None:
        metricas = []
        for columna in columnas:
            if columna == grupo:
                continue
            try:
                float(filas[0][columna])
            except (TypeError, ValueError):
                continue
            metricas.append(columna)

    metricas = list(metricas)
    faltantes = [metrica for metrica in metricas if metrica not in columnas]
    if faltantes:
        raise ValueError(f"No existen estas métricas en el CSV: {faltantes}")
    if not metricas:
        raise ValueError("No se encontraron columnas numéricas de métricas")

    resultados = []
    for fila in filas:
        fila_convertida = dict(fila)
        for metrica in metricas:
            try:
                fila_convertida[metrica] = float(fila[metrica])
            except (TypeError, ValueError) as error:
                raise ValueError(
                    f"La columna {metrica!r} contiene un valor no numérico: {fila[metrica]!r}"
                ) from error

        if "parameters" in fila_convertida:
            try:
                fila_convertida["parameters"] = json.loads(fila_convertida["parameters"])
            except (json.JSONDecodeError, TypeError):
                try:
                    fila_convertida["parameters"] = ast.literal_eval(fila_convertida["parameters"])
                except (ValueError, SyntaxError):
                    pass
        resultados.append(fila_convertida)

    for metrica in metricas:
        mejor_valor = max(fila[metrica] for fila in resultados)
        mejores = [fila for fila in resultados if fila[metrica] == mejor_valor]

        print(f"\nMejor configuración para {metrica}:")
        print(f"Valor: {mejor_valor:.6f} | Empates: {len(mejores)}")
        for numero, fila in enumerate(mejores, start=1):
            configuracion = {
                columna: fila[columna]
                for columna in columnas
                if columna not in metricas
            }
            print(f"  {numero}. {configuracion}")

    return resultados


# Cambia únicamente el nombre del archivo para analizar otro CSV.
resultados = mejores_configuraciones("config_results/hsv_experiments.csv")



Mejor configuración para map1:
Valor: 0.566667 | Empates: 5
  1. {'descriptor': 'hsv', 'bins': '16-16-8', 'parameters': {'hue_smoothing': False, 'hue_valid_weight': True, 's_min': 20, 'v_min': 40}, 'measure': 'l1'}
  2. {'descriptor': 'hsv', 'bins': '64-32-16', 'parameters': {'hue_smoothing': False, 'hue_valid_weight': False, 's_min': 60, 'v_min': 40}, 'measure': 'chi2'}
  3. {'descriptor': 'hsv', 'bins': '64-32-16', 'parameters': {'hue_smoothing': False, 'hue_valid_weight': False, 's_min': 60, 'v_min': 40}, 'measure': 'hellinger'}
  4. {'descriptor': 'hsv', 'bins': '64-32-16', 'parameters': {'hue_smoothing': True, 'hue_valid_weight': False, 's_min': 60, 'v_min': 60}, 'measure': 'l1'}
  5. {'descriptor': 'hsv', 'bins': '64-32-16', 'parameters': {'hue_smoothing': True, 'hue_valid_weight': False, 's_min': 60, 'v_min': 60}, 'measure': 'intersection'}

Mejor configuración para map5:
Valor: 0.647222 | Empates: 1
  1. {'descriptor': 'hsv', 'bins': '16-16-8', 'parameters': {'hue_smoothing': 

In [6]:
# Relacionar cada mejor resultado con la otra métrica de la misma configuración.
for metrica_principal, metrica_complementaria in (("map5", "map1"), ("map1", "map5")):
    mejor_valor = max(fila[metrica_principal] for fila in resultados)
    mejores = [
        fila for fila in resultados
        if fila[metrica_principal] == mejor_valor
    ]

    print(f"\nConfiguraciones con mejor {metrica_principal}:")
    for numero, fila in enumerate(mejores, start=1):
        print(f"  {numero}. {metrica_principal}={fila[metrica_principal]:.6f} | "
              f"{metrica_complementaria}={fila[metrica_complementaria]:.6f}")
        print(f"     descriptor={fila['descriptor']}, bins={fila['bins']}, "
              f"measure={fila['measure']}, parameters={fila['parameters']}")



Configuraciones con mejor map5:
  1. map5=0.647222 | map1=0.566667
     descriptor=hsv, bins=16-16-8, measure=l1, parameters={'hue_smoothing': False, 'hue_valid_weight': True, 's_min': 20, 'v_min': 40}

Configuraciones con mejor map1:
  1. map1=0.566667 | map5=0.647222
     descriptor=hsv, bins=16-16-8, measure=l1, parameters={'hue_smoothing': False, 'hue_valid_weight': True, 's_min': 20, 'v_min': 40}
  2. map1=0.566667 | map5=0.584444
     descriptor=hsv, bins=64-32-16, measure=chi2, parameters={'hue_smoothing': False, 'hue_valid_weight': False, 's_min': 60, 'v_min': 40}
  3. map1=0.566667 | map5=0.597222
     descriptor=hsv, bins=64-32-16, measure=hellinger, parameters={'hue_smoothing': False, 'hue_valid_weight': False, 's_min': 60, 'v_min': 40}
  4. map1=0.566667 | map5=0.601111
     descriptor=hsv, bins=64-32-16, measure=l1, parameters={'hue_smoothing': True, 'hue_valid_weight': False, 's_min': 60, 'v_min': 60}
  5. map1=0.566667 | map5=0.601111
     descriptor=hsv, bins=64-32-16,

In [8]:
# Ranking de las configuraciones ganadoras de map1 o map5 según su suma.
mejor_map1 = max(fila["map1"] for fila in resultados)
mejor_map5 = max(fila["map5"] for fila in resultados)

configuraciones_ganadoras = []
for fila in resultados:
    es_mejor_map1 = fila["map1"] == mejor_map1
    es_mejor_map5 = fila["map5"] == mejor_map5
    if es_mejor_map1 or es_mejor_map5:
        configuraciones_ganadoras.append(fila)

ranking = sorted(
    configuraciones_ganadoras,
    key=lambda fila: fila["map1"] + fila["map5"],
    reverse=True,
)

print("Ranking de las mejores configuraciones por map1 + map5:")
for posicion, fila in enumerate(ranking, start=1):
    suma = fila["map1"] + fila["map5"]
    print(
        f"{posicion:>3}. suma={suma:.6f} | "
        f"map1={fila['map1']:.6f} | map5={fila['map5']:.6f} | "
        f"descriptor={fila['descriptor']} | bins={fila['bins']} | "
        f"measure={fila['measure']} | parameters={fila['parameters']}"
    )


Ranking de las mejores configuraciones por map1 + map5:
  1. suma=1.213889 | map1=0.566667 | map5=0.647222 | descriptor=hsv | bins=16-16-8 | measure=l1 | parameters={'hue_smoothing': False, 'hue_valid_weight': True, 's_min': 20, 'v_min': 40}
  2. suma=1.167778 | map1=0.566667 | map5=0.601111 | descriptor=hsv | bins=64-32-16 | measure=l1 | parameters={'hue_smoothing': True, 'hue_valid_weight': False, 's_min': 60, 'v_min': 60}
  3. suma=1.167778 | map1=0.566667 | map5=0.601111 | descriptor=hsv | bins=64-32-16 | measure=intersection | parameters={'hue_smoothing': True, 'hue_valid_weight': False, 's_min': 60, 'v_min': 60}
  4. suma=1.163889 | map1=0.566667 | map5=0.597222 | descriptor=hsv | bins=64-32-16 | measure=hellinger | parameters={'hue_smoothing': False, 'hue_valid_weight': False, 's_min': 60, 'v_min': 40}
  5. suma=1.151111 | map1=0.566667 | map5=0.584444 | descriptor=hsv | bins=64-32-16 | measure=chi2 | parameters={'hue_smoothing': False, 'hue_valid_weight': False, 's_min': 60, 'v

In [9]:
resultados = mejores_configuraciones("config_results/rgb_experiments.csv")


Mejor configuración para map1:
Valor: 0.433333 | Empates: 1
  1. {'descriptor': 'rgb', 'bins': '8-8-8', 'parameters': {}, 'measure': 'hellinger'}

Mejor configuración para map5:
Valor: 0.450000 | Empates: 1
  1. {'descriptor': 'rgb', 'bins': '8-8-8', 'parameters': {}, 'measure': 'hellinger'}
